<a href="https://colab.research.google.com/github/sinayavarian/ML-practice/blob/main/Great_Customer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%%writefile great_customers_modeling.py
#!/usr/bin/env python3
import argparse
import warnings
from typing import Dict, Optional, Tuple, List

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, balanced_accuracy_score
from sklearn.feature_selection import SelectFromModel

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")


def data_loader(path: str) -> pd.DataFrame:
    return pd.read_csv(path)


def normalize_strings(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"": np.nan, "nan": np.nan, "None": np.nan, "NULL": np.nan, "null": np.nan})
    )


def coerce_mostly_numeric(df: pd.DataFrame, cols: List[str], threshold: float = 0.85) -> pd.DataFrame:
    for c in cols:
        if df[c].dtype == "object":
            cleaned = normalize_strings(df[c]).str.replace(",", "", regex=False)
            as_num = pd.to_numeric(cleaned, errors="coerce")
            if as_num.notna().mean() >= threshold:
                df[c] = as_num
            else:
                df[c] = cleaned
    return df


def data_cleaning(df: pd.DataFrame) -> Tuple[pd.DataFrame, str, str]:
    if df.shape[1] < 3:
        raise ValueError("Invalid dataset shape.")
    id_col = df.columns[0]
    target_col = df.columns[-1]
    df = df.dropna(how="all").copy()
    df = df.drop_duplicates(subset=[id_col], keep="first")

    t = normalize_strings(df[target_col]).str.lower()

    t = t.replace(
        {
            "great_customers": "great_customer",
            "great customer": "great_customer",
            "great": "great_customer",
            "no_great_customers": "no_great_customer",
            "no great customer": "no_great_customer",
            "not_great_customer": "no_great_customer",
            "not great": "no_great_customer",
            "nogreat": "no_great_customer",
            "0": "no_great_customer",
            "1": "great_customer",
            "false": "no_great_customer",
            "true": "great_customer",
            "no": "no_great_customer",
            "yes": "great_customer",
        }
    )

    df[target_col] = t

    vc = df[target_col].value_counts(dropna=True)
    if vc.shape[0] == 0:
        raise ValueError("No valid class labels found after normalization.")
    if vc.shape[0] > 2:
        top2 = vc.index[:2].tolist()
        df = df[df[target_col].isin(top2)].copy()

    feature_cols = [c for c in df.columns if c not in (id_col, target_col)]
    df = coerce_mostly_numeric(df, feature_cols, threshold=0.85)
    return df, id_col, target_col



def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    categorical_pipe = Pipeline(
        [("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]
    )
    return ColumnTransformer(
        [("num", numeric_pipe, num_cols), ("cat", categorical_pipe, cat_cols)],
        remainder="drop",
        sparse_threshold=0.3,
    )


def pick_primary_metric(y: np.ndarray) -> str:
    vals, counts = np.unique(y, return_counts=True)
    if len(vals) != 2:
        return "accuracy"
    ratio = counts.max() / max(counts.min(), 1)
    return "roc_auc" if ratio >= 1.5 else "f1"


def choose_feature_selector(
    X_train: pd.DataFrame, y_train: np.ndarray, preprocessor: ColumnTransformer, primary_metric: str, seed: int
) -> Tuple[str, Optional[SelectFromModel]]:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    scoring_map = {"accuracy": "accuracy", "balanced_accuracy": "balanced_accuracy", "f1": "f1", "roc_auc": "roc_auc"}
    scoring = scoring_map[primary_metric]
    baseline = LogisticRegression(max_iter=4000, solver="liblinear")
    def cv_score(pipe: Pipeline) -> float:
        return float(np.mean(cross_val_score(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)))
    best_name, best_sel, best_score = "none", None, cv_score(Pipeline([("prep", preprocessor), ("model", baseline)]))
    l1_est = LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000, random_state=seed)
    sel_l1 = SelectFromModel(l1_est, threshold="median")
    s1 = cv_score(Pipeline([("prep", preprocessor), ("fs", sel_l1), ("model", baseline)]))
    if s1 > best_score:
        best_name, best_sel, best_score = "l1_logreg", sel_l1, s1
    rf_est = RandomForestClassifier(n_estimators=400, random_state=seed, n_jobs=-1, class_weight="balanced")
    sel_rf = SelectFromModel(rf_est, threshold="median")
    s2 = cv_score(Pipeline([("prep", preprocessor), ("fs", sel_rf), ("model", baseline)]))
    if s2 > best_score:
        best_name, best_sel, best_score = "random_forest", sel_rf, s2
    return best_name, best_sel


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_score: Optional[np.ndarray]) -> Dict[str, float]:
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "roc_auc": float("nan"),
    }
    if y_score is not None:
        out["roc_auc"] = float(roc_auc_score(y_true, y_score))
    return out


def fit_predict(pipe: Pipeline, X_train: pd.DataFrame, y_train: np.ndarray, X_test: pd.DataFrame) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_score = None
    if hasattr(pipe, "predict_proba"):
        try:
            proba = pipe.predict_proba(X_test)
            if proba is not None and proba.shape[1] == 2:
                y_score = proba[:, 1]
        except Exception:
            y_score = None
    elif hasattr(pipe, "decision_function"):
        try:
            scores = pipe.decision_function(X_test)
            y_score = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
        except Exception:
            y_score = None
    return y_pred, y_score


def build_pipelines(preprocessor: ColumnTransformer, selector: Optional[SelectFromModel], seed: int) -> Dict[str, Pipeline]:
    def make(model):
        steps = [("prep", preprocessor)]
        if selector is not None:
            steps.append(("fs", selector))
        steps.append(("model", model))
        return Pipeline(steps)
    return {
        "RandomForest": make(RandomForestClassifier(n_estimators=600, random_state=seed, n_jobs=-1, class_weight="balanced")),
        "SVM": make(SVC(kernel="rbf", C=3.0, gamma="scale", probability=True, class_weight="balanced", random_state=seed)),
        "LogisticRegression": make(LogisticRegression(max_iter=6000, solver="lbfgs", class_weight="balanced")),
        "NaiveBayes": make(GaussianNB()),
        "KNN": make(KNeighborsClassifier(n_neighbors=11)),
    }


def build_voting_ensemble(preprocessor: ColumnTransformer, selector: Optional[SelectFromModel], seed: int) -> Pipeline:
    def base(model):
        steps = [("prep", preprocessor)]
        if selector is not None:
            steps.append(("fs", selector))
        steps.append(("model", model))
        return Pipeline(steps)
    lr = LogisticRegression(max_iter=6000, solver="lbfgs", class_weight="balanced")
    rf = RandomForestClassifier(n_estimators=600, random_state=seed, n_jobs=-1, class_weight="balanced")
    svm = SVC(kernel="rbf", C=3.0, gamma="scale", probability=True, class_weight="balanced", random_state=seed)
    vc = VotingClassifier(
        estimators=[("lr", base(lr)), ("rf", base(rf)), ("svm", base(svm))],
        voting="soft",
        n_jobs=-1,
    )
    return Pipeline([("ensemble", vc)])


def run(csv_path: str, test_size: float, seed: int) -> None:
    df = data_loader(csv_path)
    df, id_col, target_col = data_cleaning(df)
    X = df.drop(columns=[id_col, target_col], errors="ignore")
    labels = df[target_col].astype(str).values
    u = pd.Series(labels).value_counts().index.tolist()
    if len(u) != 2:
        raise ValueError(f"Expected 2 classes, got {len(u)}: {u}")

    a, b = u[0], u[1]
    def is_negative(s: str) -> bool:
        s = str(s).lower()
        return ("no" in s) or ("not" in s) or (s in {"0", "false"})

    neg = a if is_negative(a) and not is_negative(b) else (b if is_negative(b) and not is_negative(a) else min(a, b))
    pos = b if neg == a else a
    y = np.where(df[target_col].astype(str).values == pos, 1, 0).astype(int)

    primary_metric = pick_primary_metric(y)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, stratify=y, random_state=seed)
    preprocessor = build_preprocessor(X_train)
    fs_name, selector = choose_feature_selector(X_train, y_train, preprocessor, primary_metric, seed)
    pipes = build_pipelines(preprocessor, selector, seed)
    print(f"primary_metric={primary_metric}")
    print(f"feature_selector={fs_name}")
    best_acc = -1.0
    for name, pipe in pipes.items():
        y_pred, y_score = fit_predict(pipe, X_train, y_train, X_test)
        m = compute_metrics(y_test, y_pred, y_score)
        best_acc = max(best_acc, m["accuracy"])
        print(f"{name} accuracy={m['accuracy']:.4f} balanced_accuracy={m['balanced_accuracy']:.4f} f1={m['f1']:.4f} roc_auc={m['roc_auc']:.4f}")
    ens = build_voting_ensemble(preprocessor, selector, seed)
    y_pred_e, y_score_e = fit_predict(ens, X_train, y_train, X_test)
    me = compute_metrics(y_test, y_pred_e, y_score_e)
    print(f"Ensemble(Voting) accuracy={me['accuracy']:.4f} balanced_accuracy={me['balanced_accuracy']:.4f} f1={me['f1']:.4f} roc_auc={me['roc_auc']:.4f}")
    if me["accuracy"] > best_acc + 1e-9:
        print("accuracy_increased_with_ensemble=True")
    else:
        print("accuracy_increased_with_ensemble=False")


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--csv", type=str, default="https://raw.githubusercontent.com/subashgandyer/datasets/main/great_customers.csv")
    p.add_argument("--test_size", type=float, default=0.25)
    p.add_argument("--seed", type=int, default=42)
    return p.parse_args()


if __name__ == "__main__":
    args = parse_args()
    run(args.csv, args.test_size, args.seed)



Overwriting great_customers_modeling.py


In [6]:
!python3 great_customers_modeling.py


primary_metric=roc_auc
feature_selector=none
RandomForest accuracy=0.9079 balanced_accuracy=0.7158 f1=0.5714 roc_auc=0.9099
SVM accuracy=0.8363 balanced_accuracy=0.8042 f1=0.5578 roc_auc=0.8935
LogisticRegression accuracy=0.8265 balanced_accuracy=0.8246 f1=0.5627 roc_auc=0.9096
NaiveBayes accuracy=0.6907 balanced_accuracy=0.7720 f1=0.4369 roc_auc=0.8699
KNN accuracy=0.9014 balanced_accuracy=0.6976 f1=0.5351 roc_auc=0.8775
Ensemble(Voting) accuracy=0.8926 balanced_accuracy=0.7589 f1=0.5926 roc_auc=0.9130
accuracy_increased_with_ensemble=False
